In [24]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict
from langchain_huggingface import ChatHuggingFace,HuggingFaceEndpoint
from dotenv import load_dotenv


In [18]:

load_dotenv()
llm = HuggingFaceEndpoint(
    repo_id="deepseek-ai/DeepSeek-V3-0324",
    task="conversational",
    provider="novita",
    max_new_tokens=512,
    do_sample=False,
    temperature=0.1,
)
model = ChatHuggingFace(llm=llm)


In [19]:
result = model.invoke("Whats the capital of India");

In [20]:
result

AIMessage(content="The capital of India is **New Delhi**. It is located in the northern part of the country and serves as the administrative, political, and cultural hub of India. \n\nNew Delhi is a part of the larger **National Capital Territory (NCT) of Delhi**, which includes other cities and areas. Key landmarks in New Delhi include the **Rashtrapati Bhavan (President's House)**, **India Gate**, and the **Parliament of India**. \n\nLet me know if you'd like more details! 😊", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 108, 'prompt_tokens': 8, 'total_tokens': 116}, 'model_name': 'deepseek-ai/DeepSeek-V3-0324', 'system_fingerprint': '', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a03a33-95af-7d40-be3e-dcd81c5ee368-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 108, 'total_tokens': 116})

In [21]:
class LLMState(TypedDict):
    question:str
    answer:str

In [22]:
def llm_qa(state:LLMState)->LLMState:
    question=state['question']
    prompt = f'Answer the following question {question}'
    answer = model.invoke(prompt)
    state['answer']=answer
    return state

In [26]:
# main execution
graph = StateGraph(LLMState)

graph.add_node('llm_qa',llm_qa)

graph.add_edge(START,'llm_qa')
graph.add_edge('llm_qa',END)

workflow = graph.compile()


In [30]:
initial_state={'question':'How far is moon from earth?'}

In [31]:
final_state = workflow.invoke(initial_state)

In [32]:
final_state

{'question': 'How far is moon from earth?',
 'answer': AIMessage(content="The average distance from the **Earth to the Moon** is approximately **384,400 kilometers (238,855 miles)**.  \n\nHowever, this distance varies because the Moon follows an **elliptical orbit** around Earth:  \n- **Perigee** (closest approach): ~363,300 km (225,700 miles)  \n- **Apogee** (farthest point): ~405,500 km (252,000 miles)  \n\nThis variation affects phenomena like **supermoon** (when the Moon is closer and appears larger) and **micromoon** (when it's farther and appears smaller).  \n\nWould you like details on how this distance is measured or historical changes in the Moon's orbit?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 149, 'prompt_tokens': 14, 'total_tokens': 163}, 'model_name': 'deepseek-ai/DeepSeek-V3-0324', 'system_fingerprint': '', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a03a36-9ba8-7371-ab48-ead5f53d8030-0', tool_calls=[], invalid_tool_ca